In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# Azure Storage configuration
STORAGE_ACCOUNT = "final23"

account_key = dbutils.secrets.get(
    scope="azure-storage",
    key="storage-account-key"
)

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    account_key
)

print("ADLS configuration loaded successfully")

ADLS configuration loaded successfully


In [0]:
customers_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("abfss://landing@final23.dfs.core.windows.net/customers.csv")

inventory_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("abfss://landing@final23.dfs.core.windows.net/inventory.csv")

order_items_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("abfss://landing@final23.dfs.core.windows.net/order_items.csv")

orders_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("abfss://landing@final23.dfs.core.windows.net/orders.csv")

print("All 4 datasets loaded successfully")

All 4 datasets loaded successfully


Checking Schemas

In [0]:
print("ORDERS")
orders_df.printSchema()

print("\n ORDER ITEMS")
order_items_df.printSchema()

print("\n CUSTOMERS")
customers_df.printSchema()

print("\n INVENTORY")
inventory_df.printSchema()


 INVENTORY
root
 |-- sku_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- stock_quantity: integer (nullable = true)
 |-- reorder_level: integer (nullable = true)
 |-- unit_cost: double (nullable = true)
 |-- last_updated: timestamp (nullable = true)
 |-- load_ts: timestamp (nullable = true)



Lets check for null values

In [0]:
from pyspark.sql import functions as F

def check_nulls(df, name):
    print(f"\n {name}")
    
    result = df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ])
    
    result.show(truncate=False)


check_nulls(orders_df, "ORDERS")
check_nulls(order_items_df, "ORDER ITEMS")
check_nulls(customers_df, "CUSTOMERS")
check_nulls(inventory_df, "INVENTORY")


 ORDERS
+--------+-----------+----------+------+------------+---------------+--------------+------------+------+-------+
|order_id|customer_id|order_date|status|total_amount|discount_amount|payment_method|warehouse_id|region|load_ts|
+--------+-----------+----------+------+------------+---------------+--------------+------------+------+-------+
|0       |501        |0         |0     |0           |0              |0             |0           |0     |0      |
+--------+-----------+----------+------+------------+---------------+--------------+------------+------+-------+


 ORDER ITEMS
+-------+--------+------+------------+--------+--------+----------+----------+-------+
|item_id|order_id|sku_id|product_name|category|quantity|unit_price|line_total|load_ts|
+-------+--------+------+------------+--------+--------+----------+----------+-------+
|0      |0       |0     |0           |0       |2986    |0         |4956      |0      |
+-------+--------+------+------------+--------+--------+-------

Lets check duplicate orders

In [0]:
duplicate_orders = orders_df.groupBy("order_id") \
    .count() \
    .filter(F.col("count") > 1)

print("Number of duplicate order IDs:", duplicate_orders.count())

duplicate_orders.show(10, truncate=False)

Number of duplicate order IDs: 150
+-----------+-----+
|order_id   |count|
+-----------+-----+
|ORD00029873|2    |
|ORD00027613|2    |
|ORD00032618|2    |
|ORD00033686|2    |
|ORD00032980|2    |
|ORD00035861|2    |
|ORD00005873|2    |
|ORD00015026|2    |
|ORD00016710|2    |
|ORD00029472|2    |
+-----------+-----+
only showing top 10 rows


Lets check for invalied order status

In [0]:
orders_df.groupBy("status") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(truncate=False)

+---------+-----+
|status   |count|
+---------+-----+
|placed   |10310|
|shipped  |10290|
|delivered|10280|
|cancelled|10238|
|refunded |3046 |
|unknown  |3008 |
|PENDING  |2978 |
+---------+-----+



Lets check for order total

In [0]:
invalid_order_totals = orders_df.filter(
    F.col("total_amount") <= 0
)

print("Orders with total_amount <= 0:", invalid_order_totals.count())

invalid_order_totals.show(10, truncate=False)

Orders with total_amount <= 0: 0
+--------+-----------+----------+------+------------+---------------+--------------+------------+------+-------+
|order_id|customer_id|order_date|status|total_amount|discount_amount|payment_method|warehouse_id|region|load_ts|
+--------+-----------+----------+------+------------+---------------+--------------+------------+------+-------+
+--------+-----------+----------+------+------------+---------------+--------------+------------+------+-------+



In [0]:
invalid_quantities = order_items_df.filter(
    F.col("quantity").isNull() | (F.col("quantity") <= 0)
)

invalid_prices = order_items_df.filter(
    F.col("unit_price").isNull() | (F.col("unit_price") <= 0)
)

print("Invalid quantity rows:", invalid_quantities.count())
print("Invalid unit_price rows:", invalid_prices.count())

Invalid quantity rows: 2986
Invalid unit_price rows: 2000


In [0]:
invalid_stock = inventory_df.filter(
    F.col("stock_quantity").isNull()
)

print("NULL stock_quantity rows:", invalid_stock.count())

invalid_stock.show(10, truncate=False)

NULL stock_quantity rows: 152
+--------+--------------------------------------+--------+------------+--------------+-------------+---------+-------------------+-------------------+
|sku_id  |product_name                          |category|warehouse_id|stock_quantity|reorder_level|unit_cost|last_updated       |load_ts            |
+--------+--------------------------------------+--------+------------+--------------+-------------+---------+-------------------+-------------------+
|SKU00001|Open-architected maximized time-frame |Books   |WH-MUM      |NULL          |51           |9076.16  |2025-04-09 07:14:39|2025-04-17 06:00:00|
|SKU00034|Persistent logistical migration       |Books   |WH-DEL      |NULL          |82           |3514.11  |2025-04-07 01:39:49|2025-04-17 06:00:00|
|SKU00067|Fully-configurable stable success     |Toys    |WH-CHE      |NULL          |85           |1127.91  |2025-04-05 01:45:41|2025-04-17 06:00:00|
|SKU00100|Object-based 6thgeneration software   |Toys    |WH-DEL

In [0]:
invalid_emails = customers_df.filter(
    ~F.col("email").contains("@")
)

print("Malformed email rows:", invalid_emails.count())

invalid_emails.select(
    "customer_id",
    "email"
).show(10, truncate=False)

Malformed email rows: 0
+-----------+-----+
|customer_id|email|
+-----------+-----+
+-----------+-----+



Now that we have have validated the data lets build the landing layer

In [0]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS hive_metastore.ecommerce_landing
""")

print("Landing database created")

Landing database created


In [0]:
from pyspark.sql import functions as F

landing_path = "abfss://landing@final23.dfs.core.windows.net"

datasets = {
    "orders": f"{landing_path}/orders.csv",
    "order_items": f"{landing_path}/order_items.csv",
    "customers": f"{landing_path}/customers.csv",
    "inventory": f"{landing_path}/inventory.csv"
}

for table_name, file_path in datasets.items():

    df = spark.read \
        .option("header", "true") \
        .option("inferSchema", "false") \
        .csv(file_path)

    df = df.withColumn(
        "landing_timestamp",
        F.current_timestamp()
    ).withColumn(
        "source_file_name",
        F.lit(file_path)
    )

    df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(
            f"hive_metastore.ecommerce_landing.{table_name}"
        )

    print(f"{table_name} - Landing complete")

orders - Landing complete
order_items - Landing complete
customers - Landing complete
inventory - Landing complete


In [0]:
spark.sql("SHOW TABLES IN hive_metastore.ecommerce_landing").show(
    truncate=False
)
spark.sql("""
DESCRIBE hive_metastore.ecommerce_landing.orders
""").show(truncate=False)

+-----------------+-----------+-----------+
|database         |tableName  |isTemporary|
+-----------------+-----------+-----------+
|ecommerce_landing|customers  |false      |
|ecommerce_landing|inventory  |false      |
|ecommerce_landing|order_items|false      |
|ecommerce_landing|orders     |false      |
+-----------------+-----------+-----------+

+-----------------+---------+-------+
|col_name         |data_type|comment|
+-----------------+---------+-------+
|order_id         |string   |NULL   |
|customer_id      |string   |NULL   |
|order_date       |string   |NULL   |
|status           |string   |NULL   |
|total_amount     |string   |NULL   |
|discount_amount  |string   |NULL   |
|payment_method   |string   |NULL   |
|warehouse_id     |string   |NULL   |
|region           |string   |NULL   |
|load_ts          |string   |NULL   |
|landing_timestamp|timestamp|NULL   |
|source_file_name |string   |NULL   |
+-----------------+---------+-------+



Now that we have created landing layer lets jump to Bronze :)

In [0]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS hive_metastore.ecommerce_bronze
""")

print("Bronze database created")

Bronze database created


In [0]:
from pyspark.sql import functions as F

landing_tables = [
    "orders",
    "order_items",
    "customers",
    "inventory"
]

for table_name in landing_tables:

    df = spark.table(
        f"hive_metastore.ecommerce_landing.{table_name}"
    )

    bronze_df = df.withColumn(
        "bronze_ingestion_timestamp",
        F.current_timestamp()
    ).withColumn(
        "load_date",
        F.to_date(F.col("landing_timestamp"))
    )

    bronze_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .partitionBy("load_date") \
        .saveAsTable(
            f"hive_metastore.ecommerce_bronze.{table_name}"
        )

    print(f"{table_name} - Bronze complete")

orders - Bronze complete
order_items - Bronze complete
customers - Bronze complete
inventory - Bronze complete


In [0]:
spark.sql("""
SHOW TABLES IN hive_metastore.ecommerce_bronze
""").show(truncate=False)

+----------------+-----------+-----------+
|database        |tableName  |isTemporary|
+----------------+-----------+-----------+
|ecommerce_bronze|customers  |false      |
|ecommerce_bronze|inventory  |false      |
|ecommerce_bronze|order_items|false      |
|ecommerce_bronze|orders     |false      |
+----------------+-----------+-----------+



In [0]:
spark.sql("""
DESCRIBE hive_metastore.ecommerce_bronze.orders
""").show(truncate=False)

+--------------------------+---------+-------+
|col_name                  |data_type|comment|
+--------------------------+---------+-------+
|order_id                  |string   |NULL   |
|customer_id               |string   |NULL   |
|order_date                |string   |NULL   |
|status                    |string   |NULL   |
|total_amount              |string   |NULL   |
|discount_amount           |string   |NULL   |
|payment_method            |string   |NULL   |
|warehouse_id              |string   |NULL   |
|region                    |string   |NULL   |
|load_ts                   |string   |NULL   |
|landing_timestamp         |timestamp|NULL   |
|source_file_name          |string   |NULL   |
|bronze_ingestion_timestamp|timestamp|NULL   |
|load_date                 |date     |NULL   |
|# Partition Information   |         |       |
|# col_name                |data_type|comment|
|load_date                 |date     |NULL   |
+--------------------------+---------+-------+



Now that the Bronze is create lets jump to Silver :)

In [0]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS hive_metastore.ecommerce_silver
""")

print("Silver database created")

Silver database created


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read Bronze orders
orders_bronze = spark.table(
    "hive_metastore.ecommerce_bronze.orders"
)

# 1. Deduplicate using latest bronze ingestion

dedup_window = Window \
    .partitionBy("order_id") \
    .orderBy(
        F.col("bronze_ingestion_timestamp").desc()
    )

orders_dedup = orders_bronze \
    .withColumn("_rn", F.row_number().over(dedup_window)) \
    .filter(F.col("_rn") == 1) \
    .drop("_rn")

# 2. Cast columns to business types

orders_typed = orders_dedup.select(
    F.col("order_id").cast("string"),
    F.col("customer_id").cast("string"),
    F.col("order_date").cast("timestamp"),
    F.col("status").cast("string"),
    F.col("total_amount").cast("double"),
    F.col("discount_amount").cast("double"),
    F.col("payment_method").cast("string"),
    F.col("warehouse_id").cast("string"),
    F.col("region").cast("string"),
    F.col("load_ts").cast("timestamp"),
    F.col("landing_timestamp"),
    F.col("source_file_name"),
    F.col("bronze_ingestion_timestamp"),
    F.col("load_date")
)

print("Orders deduplicated and typed")

Orders deduplicated and typed


Lets apply Orders DQ Rules

In [0]:

# 1. Assert order_id is not NULL

null_order_ids = orders_typed.filter(
    F.col("order_id").isNull()
).count()

if null_order_ids > 0:
    raise Exception(
        f"Pipeline aborted: {null_order_ids} orders have NULL order_id"
    )

# 2. Create quarantine reason

orders_with_reason = orders_typed.withColumn(
    "quarantine_reason",
    F.when(
        ~F.col("status").isin(
            "placed",
            "shipped",
            "delivered",
            "cancelled"
        ),
        F.lit("Invalid status")
    )
    .when(
        F.col("total_amount") <= 0,
        F.lit("Non-positive total_amount")
    )
    .when(
        F.col("customer_id").isNull(),
        F.lit("NULL customer_id")
    )
)

# 3. Quarantine failed rows

orders_quarantine = orders_with_reason.filter(
    F.col("quarantine_reason").isNotNull()
)

# 4. Clean Silver orders

orders_silver = orders_with_reason.filter(
    F.col("quarantine_reason").isNull()
).drop("quarantine_reason")

print("Orders DQ processing complete")
print("Clean orders:", orders_silver.count())
print("Quarantined orders:", orders_quarantine.count())

Orders DQ processing complete
Clean orders: 40591
Quarantined orders: 9409


In [0]:
# Lets write clean orders to Silver
orders_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "hive_metastore.ecommerce_silver.orders"
    )

# Lets write rejected orders to quarantine
orders_quarantine.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "hive_metastore.ecommerce_silver.orders_quarantine"
    )

print("Silver orders written successfully")
print("Orders Silver:", orders_silver.count())
print("Orders Quarantine:", orders_quarantine.count())

Silver orders written successfully
Orders Silver: 40591
Orders Quarantine: 9409


In [0]:
# Lets read Bronze order_items
order_items_bronze = spark.table(
    "hive_metastore.ecommerce_bronze.order_items"
)

# and then cast columns to business types
order_items_typed = order_items_bronze.select(
    F.col("item_id").cast("string"),
    F.col("order_id").cast("string"),
    F.col("sku_id").cast("string"),
    F.col("product_name").cast("string"),
    F.col("category").cast("string"),
    F.col("quantity").cast("integer"),
    F.col("unit_price").cast("double"),
    F.col("line_total").cast("double"),
    F.col("load_ts").cast("timestamp"),
    F.col("landing_timestamp"),
    F.col("source_file_name"),
    F.col("bronze_ingestion_timestamp"),
    F.col("load_date")
)

print("Order items typed successfully")

Order items typed successfully


Lets apply DQ Rules to order items

In [0]:

# Create quarantine reason

order_items_with_reason = order_items_typed.withColumn(
    "quarantine_reason",
    F.when(
        F.col("quantity").isNull() | (F.col("quantity") <= 0),
        F.lit("Invalid quantity")
    )
    .when(
        F.col("unit_price").isNull() | (F.col("unit_price") <= 0),
        F.lit("Invalid unit_price")
    )
)

# Quarantine failed rows

order_items_quarantine = order_items_with_reason.filter(
    F.col("quarantine_reason").isNotNull()
)

# Clean Silver rows

order_items_silver = order_items_with_reason.filter(
    F.col("quarantine_reason").isNull()
).drop("quarantine_reason")

print("Order items DQ processing complete")
print("Clean order items:", order_items_silver.count())
print("Quarantined order items:", order_items_quarantine.count())

Order items DQ processing complete
Clean order items: 390088
Quarantined order items: 9912


In [0]:
print("SOURCE DATAFRAMES")
print("orders:", orders_df.count())
print("order_items:", order_items_df.count())
print("customers:", customers_df.count())
print("inventory:", inventory_df.count())

print("\nLANDING")
print("orders:", spark.table(
    "hive_metastore.ecommerce_landing.orders"
).count())

print("order_items:", spark.table(
    "hive_metastore.ecommerce_landing.order_items"
).count())

print("customers:", spark.table(
    "hive_metastore.ecommerce_landing.customers"
).count())

print("inventory:", spark.table(
    "hive_metastore.ecommerce_landing.inventory"
).count())

print("\nBRONZE")
print("orders:", spark.table(
    "hive_metastore.ecommerce_bronze.orders"
).count())

print("order_items:", spark.table(
    "hive_metastore.ecommerce_bronze.order_items"
).count())

print("customers:", spark.table(
    "hive_metastore.ecommerce_bronze.customers"
).count())

print("inventory:", spark.table(
    "hive_metastore.ecommerce_bronze.inventory"
).count())

SOURCE DATAFRAMES
orders: 50150
order_items: 200000
customers: 10000
inventory: 5000

LANDING
orders: 100300
order_items: 400000
customers: 20000
inventory: 10000

BRONZE
orders: 100300
order_items: 400000
customers: 20000
inventory: 10000


In [0]:
# Remove duplicated Landing tables
spark.sql("DROP TABLE IF EXISTS hive_metastore.ecommerce_landing.orders")
spark.sql("DROP TABLE IF EXISTS hive_metastore.ecommerce_landing.order_items")
spark.sql("DROP TABLE IF EXISTS hive_metastore.ecommerce_landing.customers")
spark.sql("DROP TABLE IF EXISTS hive_metastore.ecommerce_landing.inventory")

# Remove duplicated Bronze tables
spark.sql("DROP TABLE IF EXISTS hive_metastore.ecommerce_bronze.orders")
spark.sql("DROP TABLE IF EXISTS hive_metastore.ecommerce_bronze.order_items")
spark.sql("DROP TABLE IF EXISTS hive_metastore.ecommerce_bronze.customers")
spark.sql("DROP TABLE IF EXISTS hive_metastore.ecommerce_bronze.inventory")

print("Duplicated Landing and Bronze tables removed")

Duplicated Landing and Bronze tables removed


In [0]:
from pyspark.sql import functions as F

landing_path = "abfss://landing@final23.dfs.core.windows.net"

datasets = {
    "orders": f"{landing_path}/orders.csv",
    "order_items": f"{landing_path}/order_items.csv",
    "customers": f"{landing_path}/customers.csv",
    "inventory": f"{landing_path}/inventory.csv"
}

for table_name, file_path in datasets.items():

    df = spark.read \
        .option("header", "true") \
        .option("inferSchema", "false") \
        .csv(file_path)

    df = df.withColumn(
        "landing_timestamp",
        F.current_timestamp()
    ).withColumn(
        "source_file_name",
        F.lit(file_path)
    )

    df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(
            f"hive_metastore.ecommerce_landing.{table_name}"
        )

    print(f"{table_name} - Landing complete")

orders - Landing complete
order_items - Landing complete
customers - Landing complete
inventory - Landing complete


In [0]:
for table_name in datasets.keys():
    count = spark.table(
        f"hive_metastore.ecommerce_landing.{table_name}"
    ).count()
    print(f"{table_name}: {count}")

orders: 50150
order_items: 200000
customers: 10000
inventory: 5000


In [0]:
from pyspark.sql import functions as F

landing_tables = [
    "orders",
    "order_items",
    "customers",
    "inventory"
]

for table_name in landing_tables:

    df = spark.table(
        f"hive_metastore.ecommerce_landing.{table_name}"
    )

    bronze_df = df.withColumn(
        "bronze_ingestion_timestamp",
        F.current_timestamp()
    ).withColumn(
        "load_date",
        F.to_date(F.col("landing_timestamp"))
    )

    bronze_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .partitionBy("load_date") \
        .saveAsTable(
            f"hive_metastore.ecommerce_bronze.{table_name}"
        )

    print(f"{table_name} - Bronze complete")

orders - Bronze complete
order_items - Bronze complete
customers - Bronze complete
inventory - Bronze complete


In [0]:
for table_name in landing_tables:
    count = spark.table(
        f"hive_metastore.ecommerce_bronze.{table_name}"
    ).count()
    print(f"{table_name}: {count}")

orders: 50150
order_items: 200000
customers: 10000
inventory: 5000


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read corrected Bronze orders
orders_bronze = spark.table(
    "hive_metastore.ecommerce_bronze.orders"
)

# Deduplicate by order_id
dedup_window = Window \
    .partitionBy("order_id") \
    .orderBy(
        F.col("bronze_ingestion_timestamp").desc()
    )

orders_dedup = orders_bronze \
    .withColumn(
        "_rn",
        F.row_number().over(dedup_window)
    ) \
    .filter(
        F.col("_rn") == 1
    ) \
    .drop("_rn")

# Cast to business types
orders_typed = orders_dedup.select(
    F.col("order_id").cast("string"),
    F.col("customer_id").cast("string"),
    F.col("order_date").cast("timestamp"),
    F.col("status").cast("string"),
    F.col("total_amount").cast("double"),
    F.col("discount_amount").cast("double"),
    F.col("payment_method").cast("string"),
    F.col("warehouse_id").cast("string"),
    F.col("region").cast("string"),
    F.col("load_ts").cast("timestamp"),
    F.col("landing_timestamp"),
    F.col("source_file_name"),
    F.col("bronze_ingestion_timestamp"),
    F.col("load_date")
)

# Assert order_id is not NULL
null_order_ids = orders_typed.filter(
    F.col("order_id").isNull()
).count()

if null_order_ids > 0:
    raise Exception(
        f"Pipeline aborted: {null_order_ids} NULL order_id values"
    )

# Apply DQ rules
orders_with_reason = orders_typed.withColumn(
    "quarantine_reason",
    F.when(
        ~F.col("status").isin(
            "placed",
            "shipped",
            "delivered",
            "cancelled"
        ),
        "Invalid status"
    )
    .when(
        F.col("total_amount") <= 0,
        "Non-positive total_amount"
    )
    .when(
        F.col("customer_id").isNull(),
        "NULL customer_id"
    )
)

# Split clean and rejected records
orders_quarantine = orders_with_reason.filter(
    F.col("quarantine_reason").isNotNull()
)

orders_silver = orders_with_reason.filter(
    F.col("quarantine_reason").isNull()
).drop("quarantine_reason")

print("Clean orders:", orders_silver.count())
print("Quarantined orders:", orders_quarantine.count())

Clean orders: 40591
Quarantined orders: 9409


In [0]:
orders_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "hive_metastore.ecommerce_silver.orders"
    )

orders_quarantine.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "hive_metastore.ecommerce_silver.orders_quarantine"
    )

print("Corrected Silver Orders written")

Corrected Silver Orders written


In [0]:
from pyspark.sql import functions as F

customers_bronze = spark.table(
    "hive_metastore.ecommerce_bronze.customers"
)

customers_typed = customers_bronze.select(
    F.col("customer_id").cast("string"),
    F.col("first_name").cast("string"),
    F.col("last_name").cast("string"),
    F.col("email").cast("string"),
    F.col("phone").cast("long"),
    F.col("city").cast("string"),
    F.col("state").cast("string"),
    F.col("region").cast("string"),
    F.col("signup_date").cast("timestamp"),
    F.col("is_active").cast("integer"),
    F.col("load_ts").cast("timestamp"),
    F.col("landing_timestamp"),
    F.col("source_file_name"),
    F.col("bronze_ingestion_timestamp"),
    F.col("load_date")
)

print("Customers typed successfully")
print("Customer rows:", customers_typed.count())

Customers typed successfully
Customer rows: 10000


In [0]:
order_items_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "hive_metastore.ecommerce_silver.order_items"
    )

order_items_quarantine.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "hive_metastore.ecommerce_silver.order_items_quarantine"
    )

print("Silver Order Items written successfully")

Silver Order Items written successfully


In [0]:
order_items_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "hive_metastore.ecommerce_silver.order_items"
    )

order_items_quarantine.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "hive_metastore.ecommerce_silver.order_items_quarantine"
    )

print("Silver Order Items written successfully")

Silver Order Items written successfully


Lets write Customers using SCD Type-1 MERGE

In [0]:
customers_target = "hive_metastore.ecommerce_silver.customers"

customers_typed.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(customers_target)

print("Silver customers created")
print("Customer rows:", customers_typed.count())

Silver customers created
Customer rows: 10000


Lets prepare the filter Inventory

In [0]:
from pyspark.sql import functions as F

# Read Bronze inventory
inventory_bronze = spark.table(
    "hive_metastore.ecommerce_bronze.inventory"
)

# Cast columns to business types
inventory_typed = inventory_bronze.select(
    F.col("sku_id").cast("string"),
    F.col("product_name").cast("string"),
    F.col("category").cast("string"),
    F.col("warehouse_id").cast("string"),
    F.col("stock_quantity").cast("integer"),
    F.col("reorder_level").cast("integer"),
    F.col("unit_cost").cast("double"),
    F.col("last_updated").cast("timestamp"),
    F.col("load_ts").cast("timestamp"),
    F.col("landing_timestamp"),
    F.col("source_file_name"),
    F.col("bronze_ingestion_timestamp"),
    F.col("load_date")
)

# Drop rows where stock_quantity is NULL
inventory_silver = inventory_typed.filter(
    F.col("stock_quantity").isNotNull()
)

print("Inventory DQ processing complete")
print("Bronze inventory:", inventory_typed.count())
print("Silver inventory:", inventory_silver.count())
print(
    "Dropped NULL stock rows:",
    inventory_typed.count() - inventory_silver.count()
)

Inventory DQ processing complete
Bronze inventory: 5000
Silver inventory: 4848
Dropped NULL stock rows: 152


In [0]:
inventory_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "hive_metastore.ecommerce_silver.inventory"
    )

print("Silver inventory written successfully")
print("Inventory rows:", inventory_silver.count())

Silver inventory written successfully
Inventory rows: 4848


Now that ive completed the landing, bronze, silver lets jump to gold :)

In [0]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS hive_metastore.ecommerce_gold
""")

print("Gold database created")

Gold database created


Lets calculate daily revenue by category

In [0]:
from pyspark.sql import functions as F

# Read Silver tables
orders_silver = spark.table(
    "hive_metastore.ecommerce_silver.orders"
)

order_items_silver = spark.table(
    "hive_metastore.ecommerce_silver.order_items"
)

# Join orders with their items
revenue_base = orders_silver.alias("o").join(
    order_items_silver.alias("oi"),
    F.col("o.order_id") == F.col("oi.order_id"),
    "inner"
)

# Build daily revenue metrics
daily_revenue = revenue_base.groupBy(
    F.to_date(F.col("o.order_date")).alias("order_date"),
    F.col("o.region").alias("region"),
    F.col("oi.category").alias("category")
).agg(
    F.sum(F.col("oi.line_total")).alias("total_revenue"),
    F.countDistinct(F.col("o.order_id")).alias("order_count"),
    F.avg(F.col("o.total_amount")).alias("average_order_value")
)

print("Daily revenue calculated")
print("Rows:", daily_revenue.count())

daily_revenue.show(10, truncate=False)

Daily revenue calculated
Rows: 4240
+----------+-------+-----------+------------------+-----------+-------------------+
|order_date|region |category   |total_revenue     |order_count|average_order_value|
+----------+-------+-----------+------------------+-----------+-------------------+
|2025-03-26|North  |Sports     |1356553.9900000002|30         |22300.357500000002 |
|2025-01-06|North  |Grocery    |956732.45         |25         |17718.069411764707 |
|2025-02-04|East   |Beauty     |1327948.66        |29         |26541.135714285716 |
|2025-03-16|North  |Sports     |1027376.3         |17         |20987.37761904762  |
|2025-03-10|West   |Clothing   |1273506.23        |25         |23916.11           |
|2025-01-06|North  |Books      |1367048.9         |25         |24902.08441176471  |
|2025-01-04|North  |Grocery    |2239033.63        |38         |28841.00672727273  |
|2025-03-26|East   |Electronics|2000074.2799999998|35         |27022.563720930233 |
|2025-04-08|South  |Grocery    |1359283.

In [0]:
daily_revenue.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "hive_metastore.ecommerce_gold.daily_revenue"
    )

print("Gold daily_revenue written successfully")

print(
    "Total revenue:",
    daily_revenue.agg(
        F.sum("total_revenue")
    ).first()[0]
)

print(
    "Total groups:",
    daily_revenue.count()
)

Gold daily_revenue written successfully
Total revenue: 6590818452.869975
Total groups: 4240


Gold Fulfillment kpi calculation

In [0]:
from pyspark.sql import functions as F

# Read Silver orders
orders_silver = spark.table(
    "hive_metastore.ecommerce_silver.orders"
)

# Calculate fulfillment KPIs
fulfillment_kpi = orders_silver.groupBy(
    F.to_date(F.col("order_date")).alias("order_date"),
    F.col("warehouse_id").alias("warehouse_id"),
    F.col("region").alias("region")
).agg(
    (
        F.sum(
            F.when(F.col("status") == "delivered", 1).otherwise(0)
        ) / F.count("*") * 100
    ).alias("delivery_rate_pct"),

    (
        F.sum(
            F.when(F.col("status") == "cancelled", 1).otherwise(0)
        ) / F.count("*") * 100
    ).alias("cancellation_rate_pct"),

    (
        F.sum(
            F.when(F.col("status") == "shipped", 1).otherwise(0)
        ) / F.count("*") * 100
    ).alias("shipment_rate_pct")
)

print("Fulfillment KPI calculated")
print("Rows:", fulfillment_kpi.count())

fulfillment_kpi.show(10, truncate=False)

Fulfillment KPI calculated
Rows: 2650
+----------+------------+-------+------------------+---------------------+------------------+
|order_date|warehouse_id|region |delivery_rate_pct |cancellation_rate_pct|shipment_rate_pct |
+----------+------------+-------+------------------+---------------------+------------------+
|2025-01-31|WH-MUM      |North  |15.0              |40.0                 |20.0              |
|2025-02-20|WH-DEL      |East   |36.36363636363637 |27.27272727272727    |9.090909090909092 |
|2025-02-17|WH-HYD      |West   |23.076923076923077|7.6923076923076925   |30.76923076923077 |
|2025-03-18|WH-HYD      |West   |22.727272727272727|22.727272727272727   |27.27272727272727 |
|2025-01-29|WH-MUM      |West   |14.285714285714285|35.714285714285715   |0.0               |
|2025-01-26|WH-CHE      |East   |44.44444444444444 |11.11111111111111    |33.33333333333333 |
|2025-03-16|WH-BLR      |South  |22.22222222222222 |33.33333333333333    |16.666666666666664|
|2025-04-02|WH-HYD    

In [0]:
fulfillment_kpi.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "hive_metastore.ecommerce_gold.fulfillment_kpi"
    )

print("Gold fulfillment_kpi written successfully")

print(
    "Total KPI groups:",
    fulfillment_kpi.count()
)

print(
    "Average delivery rate:",
    fulfillment_kpi.agg(
        F.avg("delivery_rate_pct")
    ).first()[0]
)

print(
    "Average cancellation rate:",
    fulfillment_kpi.agg(
        F.avg("cancellation_rate_pct")
    ).first()[0]
)

print(
    "Average shipment rate:",
    fulfillment_kpi.agg(
        F.avg("shipment_rate_pct")
    ).first()[0]
)

Gold fulfillment_kpi written successfully
Total KPI groups: 2650
Average delivery rate: 24.96427882410163
Average cancellation rate: 24.98759842038841
Average shipment rate: 24.989913045911806


Lets build the function to calculate 30 days demand<br>
How i am doing it 
30-day demand
= SUM(quantity sold)
  during the 30 days ending at
  the latest order date in the dataset

In [0]:
from pyspark.sql import functions as F

# Read Silver tables
inventory_silver = spark.table(
    "hive_metastore.ecommerce_silver.inventory"
)

orders_silver = spark.table(
    "hive_metastore.ecommerce_silver.orders"
)

order_items_silver = spark.table(
    "hive_metastore.ecommerce_silver.order_items"
)

# Find the latest order date in the dataset
max_order_date = orders_silver.agg(
    F.max("order_date")
).first()[0]

print("Latest order date:", max_order_date)

# Calculate 30-day demand per SKU
demand_30d = (
    order_items_silver.alias("oi")
    .join(
        orders_silver.alias("o"),
        F.col("oi.order_id") == F.col("o.order_id"),
        "inner"
    )
    .filter(
        F.col("o.order_date") >=
        F.lit(max_order_date) - F.expr("INTERVAL 30 DAYS")
    )
    .groupBy(
        F.col("oi.sku_id")
    )
    .agg(
        F.sum("oi.quantity").alias("demand_30d")
    )
)

# Join demand with inventory
inventory_health = (
    inventory_silver.alias("i")
    .join(
        demand_30d.alias("d"),
        F.col("i.sku_id") == F.col("d.sku_id"),
        "left"
    )
    .select(
        F.col("i.sku_id"),
        F.col("i.product_name"),
        F.col("i.category"),
        F.col("i.warehouse_id"),
        F.col("i.stock_quantity"),
        F.col("i.reorder_level"),
        F.col("i.unit_cost"),
        F.coalesce(
            F.col("d.demand_30d"),
            F.lit(0)
        ).alias("demand_30d")
    )
)

print("Inventory demand calculated")
print("SKUs:", inventory_health.count())

inventory_health.show(10, truncate=False)

Latest order date: 2025-04-16 23:55:50
Inventory demand calculated
SKUs: 4848
+--------+------------------------------------------+--------+------------+--------------+-------------+---------+----------+
|sku_id  |product_name                              |category|warehouse_id|stock_quantity|reorder_level|unit_cost|demand_30d|
+--------+------------------------------------------+--------+------------+--------------+-------------+---------+----------+
|SKU00009|Integrated human-resource solution        |Books   |WH-HYD      |1580          |86           |179.72   |62        |
|SKU00003|Multi-lateral dynamic utilization         |Beauty  |WH-DEL      |1798          |90           |8386.89  |57        |
|SKU00002|Ergonomic empowering workforce            |Clothing|WH-DEL      |557           |86           |6597.28  |58        |
|SKU00007|Future-proofed homogeneous challenge      |Clothing|WH-CHE      |1481          |42           |12380.98 |63        |
|SKU00012|Function-based eco-centric fun

Lets add stock status and reorder flag

In [0]:
inventory_health = inventory_health.withColumn(
    "stock_status",
    F.when(
        F.col("stock_quantity") == 0,
        "stockout"
    )
    .when(
        F.col("stock_quantity") < F.col("reorder_level"),
        "below_reorder"
    )
    .when(
        F.col("stock_quantity") > 2 * F.col("reorder_level"),
        "overstock"
    )
    .otherwise(
        "healthy"
    )
)

inventory_health = inventory_health.withColumn(
    "reorder_flag",
    F.when(
        F.col("stock_quantity") <= F.col("reorder_level"),
        True
    ).otherwise(False)
)

print("Inventory health calculated")

inventory_health.select(
    "sku_id",
    "stock_quantity",
    "reorder_level",
    "demand_30d",
    "stock_status",
    "reorder_flag"
).show(10, truncate=False)

Inventory health calculated
+--------+--------------+-------------+----------+-------------+------------+
|sku_id  |stock_quantity|reorder_level|demand_30d|stock_status |reorder_flag|
+--------+--------------+-------------+----------+-------------+------------+
|SKU00009|1580          |86           |62        |overstock    |false       |
|SKU00003|1798          |90           |57        |overstock    |false       |
|SKU00002|557           |86           |58        |overstock    |false       |
|SKU00007|1481          |42           |63        |overstock    |false       |
|SKU00012|568           |81           |51        |overstock    |false       |
|SKU00005|76            |99           |32        |below_reorder|true        |
|SKU00006|1225          |55           |22        |overstock    |false       |
|SKU00008|797           |16           |34        |overstock    |false       |
|SKU00011|626           |84           |49        |overstock    |false       |
|SKU00010|1273          |93         

In [0]:
inventory_health.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "hive_metastore.ecommerce_gold.inventory_health"
    )

print("Gold inventory_health written successfully")

print(
    "Total SKUs:",
    inventory_health.count()
)

print(
    "Stockouts:",
    inventory_health.filter(
        F.col("stock_status") == "stockout"
    ).count()
)

print(
    "Below reorder:",
    inventory_health.filter(
        F.col("stock_status") == "below_reorder"
    ).count()
)

print(
    "Overstock:",
    inventory_health.filter(
        F.col("stock_status") == "overstock"
    ).count()
)

print(
    "Total inventory value:",
    inventory_health.agg(
        F.sum(
            F.col("stock_quantity") * F.col("unit_cost")
        )
    ).first()[0]
)

Gold inventory_health written successfully
Total SKUs: 4848
Stockouts: 2
Below reorder: 144
Overstock: 4562
Total inventory value: 36597321257.46


Calculate ustomer metrics

In [0]:
from pyspark.sql import functions as F

# Read Silver tables
orders_silver = spark.table(
    "hive_metastore.ecommerce_silver.orders"
)

customers_silver = spark.table(
    "hive_metastore.ecommerce_silver.customers"
)

# Latest order date in the dataset
max_order_date = orders_silver.agg(
    F.max("order_date")
).first()[0]

print("Latest order date:", max_order_date)

# Customer-level lifetime metrics

customer_metrics = orders_silver.groupBy(
    "customer_id"
).agg(
    F.sum("total_amount").alias("lifetime_spend"),
    F.countDistinct("order_id").alias("order_frequency"),
    F.max("order_date").alias("last_order_date")
)

# Calculate recency

customer_metrics = customer_metrics.withColumn(
    "recency_days",
    F.datediff(
        F.lit(max_order_date),
        F.to_date("last_order_date")
    )
)

# Add customer information

customer_ltv = customers_silver.alias("c").join(
    customer_metrics.alias("m"),
    F.col("c.customer_id") == F.col("m.customer_id"),
    "left"
).select(
    F.col("c.customer_id"),
    F.col("c.first_name"),
    F.col("c.last_name"),
    F.col("c.email"),
    F.col("c.region"),
    F.col("c.is_active"),
    F.coalesce(
        F.col("m.lifetime_spend"),
        F.lit(0.0)
    ).alias("lifetime_spend"),
    F.coalesce(
        F.col("m.order_frequency"),
        F.lit(0)
    ).alias("order_frequency"),
    F.coalesce(
        F.col("m.recency_days"),
        F.lit(-1)
    ).alias("recency_days")
)

print("Customer LTV metrics calculated")
print("Customers:", customer_ltv.count())

customer_ltv.show(10, truncate=False)

Latest order date: 2025-04-16 23:55:50
Customer LTV metrics calculated
Customers: 10000
+-----------+----------+---------+-----------------------+-------+---------+------------------+---------------+------------+
|customer_id|first_name|last_name|email                  |region |is_active|lifetime_spend    |order_frequency|recency_days|
+-----------+----------+---------+-----------------------+-------+---------+------------------+---------------+------------+
|CUST000001 |Liam      |Chaudry  |Udant@                 |North  |1        |159387.89         |5              |14          |
|CUST000002 |Arunima   |Ahuja    |ckannan@example.net    |South  |1        |223498.9          |6              |56          |
|CUST000003 |Kritika   |Brar     |caleb78@example.org    |North  |1        |77470.31999999999 |4              |22          |
|CUST000004 |Isha      |Kadakia  |sudiksha52@example.com |Central|1        |75311.87999999999 |3              |15          |
|CUST000005 |Nandini   |Loyal    |kar

In [0]:
customer_ltv = customer_ltv.withColumn(
    "customer_segment",
    F.when(
        F.col("lifetime_spend") >= 200000,
        "VIP"
    )
    .when(
        F.col("lifetime_spend") >= 100000,
        "High Value"
    )
    .when(
        F.col("lifetime_spend") >= 50000,
        "Mid Value"
    )
    .otherwise(
        "Low Value"
    )
)

print("Customer segmentation complete")

customer_ltv.groupBy(
    "customer_segment"
).count().orderBy(
    F.desc("count")
).show()

Customer segmentation complete
+----------------+-----+
|customer_segment|count|
+----------------+-----+
|      High Value| 4069|
|       Mid Value| 3332|
|       Low Value| 2011|
|             VIP|  588|
+----------------+-----+



In [0]:
customer_ltv.show(10, truncate=False)

+-----------+----------+---------+-----------------------+-------+---------+------------------+---------------+------------+----------------+
|customer_id|first_name|last_name|email                  |region |is_active|lifetime_spend    |order_frequency|recency_days|customer_segment|
+-----------+----------+---------+-----------------------+-------+---------+------------------+---------------+------------+----------------+
|CUST000001 |Liam      |Chaudry  |Udant@                 |North  |1        |159387.89         |5              |14          |High Value      |
|CUST000002 |Arunima   |Ahuja    |ckannan@example.net    |South  |1        |223498.9          |6              |56          |VIP             |
|CUST000003 |Kritika   |Brar     |caleb78@example.org    |North  |1        |77470.31999999999 |4              |22          |Mid Value       |
|CUST000004 |Isha      |Kadakia  |sudiksha52@example.com |Central|1        |75311.87999999999 |3              |15          |Mid Value       |
|CUST0

In [0]:
customer_ltv.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "hive_metastore.ecommerce_gold.customer_ltv"
    )

print("Gold customer_ltv written successfully")

# Total customers
total_customers = customer_ltv.count()

# Active customer percentage
active_pct = (
    customer_ltv
    .agg(
        F.avg(
            F.when(F.col("is_active") == 1, 1).otherwise(0)
        ) * 100
    )
    .first()[0]
)

# Average lifetime value
average_ltv = customer_ltv.agg(
    F.avg("lifetime_spend")
).first()[0]

print("Total customers:", total_customers)
print("Active customer %:", active_pct)
print("Average LTV:", average_ltv)

print("\nSegment breakdown:")
customer_ltv.groupBy(
    "customer_segment"
).count().orderBy(
    F.desc("count")
).show()

Gold customer_ltv written successfully
Total customers: 10000
Active customer %: 74.28
Average LTV: 101675.46587800053

Segment breakdown:
+----------------+-----+
|customer_segment|count|
+----------------+-----+
|      High Value| 4069|
|       Mid Value| 3332|
|       Low Value| 2011|
|             VIP|  588|
+----------------+-----+



Reconciliation

In [0]:
from pyspark.sql import functions as F

# Collect row counts from every layer

row_counts = [
    ("landing", "orders",
     spark.table("hive_metastore.ecommerce_landing.orders").count()),

    ("landing", "order_items",
     spark.table("hive_metastore.ecommerce_landing.order_items").count()),

    ("landing", "customers",
     spark.table("hive_metastore.ecommerce_landing.customers").count()),

    ("landing", "inventory",
     spark.table("hive_metastore.ecommerce_landing.inventory").count()),

    ("bronze", "orders",
     spark.table("hive_metastore.ecommerce_bronze.orders").count()),

    ("bronze", "order_items",
     spark.table("hive_metastore.ecommerce_bronze.order_items").count()),

    ("bronze", "customers",
     spark.table("hive_metastore.ecommerce_bronze.customers").count()),

    ("bronze", "inventory",
     spark.table("hive_metastore.ecommerce_bronze.inventory").count()),

    ("silver", "orders",
     spark.table("hive_metastore.ecommerce_silver.orders").count()),

    ("silver", "order_items",
     spark.table("hive_metastore.ecommerce_silver.order_items").count()),

    ("silver", "customers",
     spark.table("hive_metastore.ecommerce_silver.customers").count()),

    ("silver", "inventory",
     spark.table("hive_metastore.ecommerce_silver.inventory").count()),

    ("silver", "orders_quarantine",
     spark.table("hive_metastore.ecommerce_silver.orders_quarantine").count()),

    ("silver", "order_items_quarantine",
     spark.table("hive_metastore.ecommerce_silver.order_items_quarantine").count()),

    ("gold", "daily_revenue",
     spark.table("hive_metastore.ecommerce_gold.daily_revenue").count()),

    ("gold", "fulfillment_kpi",
     spark.table("hive_metastore.ecommerce_gold.fulfillment_kpi").count()),

    ("gold", "inventory_health",
     spark.table("hive_metastore.ecommerce_gold.inventory_health").count()),

    ("gold", "customer_ltv",
     spark.table("hive_metastore.ecommerce_gold.customer_ltv").count())
]

reconciliation_row_counts = spark.createDataFrame(
    row_counts,
    ["layer", "table_name", "row_count"]
).withColumn(
    "captured_at",
    F.current_timestamp()
)

reconciliation_row_counts.show(
    100,
    truncate=False
)

+-------+----------------------+---------+--------------------------+
|layer  |table_name            |row_count|captured_at               |
+-------+----------------------+---------+--------------------------+
|landing|orders                |50150    |2026-08-13 19:30:21.309622|
|landing|order_items           |200000   |2026-08-13 19:30:21.309622|
|landing|customers             |10000    |2026-08-13 19:30:21.309622|
|landing|inventory             |5000     |2026-08-13 19:30:21.309622|
|bronze |orders                |50150    |2026-08-13 19:30:21.309622|
|bronze |order_items           |200000   |2026-08-13 19:30:21.309622|
|bronze |customers             |10000    |2026-08-13 19:30:21.309622|
|bronze |inventory             |5000     |2026-08-13 19:30:21.309622|
|silver |orders                |40591    |2026-08-13 19:30:21.309622|
|silver |order_items           |195044   |2026-08-13 19:30:21.309622|
|silver |customers             |10000    |2026-08-13 19:30:21.309622|
|silver |inventory  

In [0]:
reconciliation_row_counts.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "hive_metastore.ecommerce_gold.reconciliation_row_counts"
    )

print("Reconciliation row counts written successfully")

Reconciliation row counts written successfully


In [0]:
from pyspark.sql import functions as F

# Orders DQ summary

orders_bronze_count = spark.table(
    "hive_metastore.ecommerce_bronze.orders"
).count()

orders_silver_count = spark.table(
    "hive_metastore.ecommerce_silver.orders"
).count()

orders_quarantine_count = spark.table(
    "hive_metastore.ecommerce_silver.orders_quarantine"
).count()


# Order Items DQ summary

items_bronze_count = spark.table(
    "hive_metastore.ecommerce_bronze.order_items"
).count()

items_silver_count = spark.table(
    "hive_metastore.ecommerce_silver.order_items"
).count()

items_quarantine_count = spark.table(
    "hive_metastore.ecommerce_silver.order_items_quarantine"
).count()


# Create reconciliation rows

dq_data = [
    (
        "orders",
        orders_bronze_count,
        orders_silver_count,
        orders_quarantine_count
    ),
    (
        "order_items",
        items_bronze_count,
        items_silver_count,
        items_quarantine_count
    )
]

reconciliation_dq_summary = spark.createDataFrame(
    dq_data,
    [
        "table_name",
        "bronze_row_count",
        "silver_row_count",
        "quarantined_rows"
    ]
).withColumn(
    "pass_rate_pct",
    F.round(
        F.col("silver_row_count") /
        F.col("bronze_row_count") * 100,
        2
    )
).withColumn(
    "quarantine_rate_pct",
    F.round(
        F.col("quarantined_rows") /
        F.col("bronze_row_count") * 100,
        2
    )
).withColumn(
    "captured_at",
    F.current_timestamp()
)

reconciliation_dq_summary.show(
    truncate=False
)

+-----------+----------------+----------------+----------------+-------------+-------------------+--------------------------+
|table_name |bronze_row_count|silver_row_count|quarantined_rows|pass_rate_pct|quarantine_rate_pct|captured_at               |
+-----------+----------------+----------------+----------------+-------------+-------------------+--------------------------+
|orders     |50150           |40591           |9409            |80.94        |18.76              |2026-08-13 19:31:40.245393|
|order_items|200000          |195044          |4956            |97.52        |2.48               |2026-08-13 19:31:40.245393|
+-----------+----------------+----------------+----------------+-------------+-------------------+--------------------------+



In [0]:
reconciliation_dq_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "hive_metastore.ecommerce_gold.reconciliation_dq_summary"
    )

print("Reconciliation DQ summary written successfully")

Reconciliation DQ summary written successfully


In [0]:

# FINAL PIPELINE VERIFICATION

expected_tables = {
    "ecommerce_landing": {
        "orders": 50150,
        "order_items": 200000,
        "customers": 10000,
        "inventory": 5000
    },

    "ecommerce_bronze": {
        "orders": 50150,
        "order_items": 200000,
        "customers": 10000,
        "inventory": 5000
    },

    "ecommerce_silver": {
        "orders": 40591,
        "order_items": 195044,
        "customers": 10000,
        "inventory": 4848,
        "orders_quarantine": 9409,
        "order_items_quarantine": 4956
    },

    "ecommerce_gold": {
        "daily_revenue": 4240,
        "fulfillment_kpi": 2650,
        "inventory_health": 4848,
        "customer_ltv": 10000,
        "reconciliation_row_counts": 18,
        "reconciliation_dq_summary": 2
    }
}

all_passed = True

print("=" * 60)
print("FINAL PIPELINE VERIFICATION")
print("=" * 60)

for database, tables in expected_tables.items():

    print(f"\n{database}")

    for table, expected_count in tables.items():

        full_name = f"hive_metastore.{database}.{table}"

        try:
            actual_count = spark.table(full_name).count()

            status = "PASS" if actual_count == expected_count else "FAIL"

            if status == "FAIL":
                all_passed = False

            print(
                f"{status}: {table:<30} "
                f"Expected={expected_count:<8} "
                f"Actual={actual_count}"
            )

        except Exception as e:
            all_passed = False
            print(f"FAIL: {table} → TABLE NOT FOUND")

print("\n" + "=" * 60)

if all_passed:
    print(":) PIPELINE VERIFICATION PASSED")
    print("All Landing, Bronze, Silver and Gold tables are correct.")
else:
    print(":( PIPELINE VERIFICATION FAILED")
    print("Review the failed table(s) above.")


FINAL PIPELINE VERIFICATION

ecommerce_landing
PASS: orders                         Expected=50150    Actual=50150
PASS: order_items                    Expected=200000   Actual=200000
PASS: customers                      Expected=10000    Actual=10000
PASS: inventory                      Expected=5000     Actual=5000

ecommerce_bronze
PASS: orders                         Expected=50150    Actual=50150
PASS: order_items                    Expected=200000   Actual=200000
PASS: customers                      Expected=10000    Actual=10000
PASS: inventory                      Expected=5000     Actual=5000

ecommerce_silver
PASS: orders                         Expected=40591    Actual=40591
PASS: order_items                    Expected=195044   Actual=195044
PASS: customers                      Expected=10000    Actual=10000
PASS: inventory                      Expected=4848     Actual=4848
PASS: orders_quarantine              Expected=9409     Actual=9409
PASS: order_items_quarantine      